# Subsidiary Data Analysis
Analyze parsed subsidiary data from SEC filings

## 0. Setup

In [1]:
import polars as pl
from pathlib import Path

# Load CSV
csv_path = Path("../../../../subsidiaries_SUCCESS.csv")
df = pl.read_csv(csv_path, schema_overrides={"Ownership": pl.Float64})

print(f"Total rows: {len(df):,}")
print(f"Columns: {df.columns}")
df.head()

Total rows: 180,570
Columns: ['Accession', 'URL', 'SubsidiaryId', 'Subsidiary', 'Jurisdiction', 'NestingLevel', 'ParentName', 'ParentId', 'Ownership', 'Footnotes']


Accession,URL,SubsidiaryId,Subsidiary,Jurisdiction,NestingLevel,ParentName,ParentId,Ownership,Footnotes
str,str,str,str,str,i64,str,str,f64,str
"""`000095017025038693""","""https://www.sec.gov/Archives/e…","""b91a4c12-0b65-530a-bb38-31a138…","""Medallion Funding LLC""","""New York""",0,null,"""6bd69361-5068-5121-9d53-9dd804…",100.0,""""""
"""`000095017025038693""","""https://www.sec.gov/Archives/e…","""c812f033-3f07-594d-945e-e01923…","""Medallion Capital, Inc.""","""Minnesota""",0,null,"""6bd69361-5068-5121-9d53-9dd804…",100.0,""""""
"""`000095017025038693""","""https://www.sec.gov/Archives/e…","""6c9072d2-14b2-52a5-8d54-631bf4…","""Freshstart Venture Capital Cor…","""New York""",0,null,"""6bd69361-5068-5121-9d53-9dd804…",100.0,""""""
"""`000095017025038693""","""https://www.sec.gov/Archives/e…","""23a12c5a-d217-5efe-9cb6-0d74cf…","""Medallion Bank""","""Utah""",0,null,"""6bd69361-5068-5121-9d53-9dd804…",100.0,""""""
"""`000100022825000014""","""https://www.sec.gov/Archives/e…","""a7a40ef8-7c84-5a0e-8e70-ca3aef…","""ACE Surgical Supply Co., Inc.""","""Massachusetts""",0,null,"""0644a2d7-c768-5a0d-aca8-83406d…",100.0,""""""


In [2]:
# Basic stats
df.describe()

statistic,Accession,URL,SubsidiaryId,Subsidiary,Jurisdiction,NestingLevel,ParentName,ParentId,Ownership,Footnotes
str,str,str,str,str,str,f64,str,str,f64,str
"""count""","""180570""","""180570""","""180570""","""180570""","""180570""",180570.0,"""3816""","""180570""",179945.0,"""180570"""
"""null_count""","""0""","""0""","""0""","""0""","""0""",0.0,"""176754""","""0""",625.0,"""0"""
"""mean""",null,null,null,null,null,0.055735,null,null,99.48815,null
"""std""",null,null,null,null,null,0.363866,null,null,5.49318,null
"""min""","""`000000248825000012""","""https://www.sec.gov/Archives/e…","""00005b18-bbee-5662-b85d-d58635…","""""Abbott Laboratories Baltics""""","""""ZIM Russia"" Closed Joint-Stoc…",0.0,"""0896800 B.C. Ltd.""","""002b5809-de06-5c7c-a24e-82476e…",0.0,""""""
"""25%""",null,null,null,null,null,0.0,null,null,100.0,null
"""50%""",null,null,null,null,null,0.0,null,null,100.0,null
"""75%""",null,null,null,null,null,0.0,null,null,100.0,null
"""max""","""`000207709625000107""","""https://www.sec.gov/Archives/e…","""fffff536-ec32-5f41-8f42-48485d…","""深圳前海豐泰仁匯健康科技有限公司""","""•Luxembourg""",8.0,"""◦Kanawha River Terminals LLC""","""ffef8561-669b-5240-b1f5-b2ac5d…",100.0,"""99"""


In [3]:
# Query rows for a specific accession
accession_to_find = "`000162828025008991"
df.filter(pl.col("Accession") == accession_to_find).head(10)

Accession,URL,SubsidiaryId,Subsidiary,Jurisdiction,NestingLevel,ParentName,ParentId,Ownership,Footnotes
str,str,str,str,str,i64,str,str,f64,str
"""`000162828025008991""","""https://www.sec.gov/Archives/e…","""96364b78-ea28-5a0f-8b81-299a97…","""Provident Financial Services, …","""New Jersey""",0,null,"""6211cfe6-9e02-5941-a2bb-1c8107…",100.0,""""""
"""`000162828025008991""","""https://www.sec.gov/Archives/e…","""63913fc1-2756-5fcc-a0a6-5f893f…","""Sussex Capital Trust II""","""Delaware""",0,null,"""6211cfe6-9e02-5941-a2bb-1c8107…",100.0,""""""
"""`000162828025008991""","""https://www.sec.gov/Archives/e…","""be8d5cfd-0587-52ce-9957-eaee24…","""1st Constitution Capital Trust…","""Delaware""",0,null,"""6211cfe6-9e02-5941-a2bb-1c8107…",100.0,""""""
"""`000162828025008991""","""https://www.sec.gov/Archives/e…","""1f14f247-ec43-5996-8baa-d1a8f6…","""Lakeland Bancorp Capital Trust…","""Delaware""",0,null,"""6211cfe6-9e02-5941-a2bb-1c8107…",100.0,""""""
"""`000162828025008991""","""https://www.sec.gov/Archives/e…","""f595ab0b-064b-556b-820c-a7d03f…","""Lakeland Bancorp Capital Trust…","""Delaware""",0,null,"""6211cfe6-9e02-5941-a2bb-1c8107…",100.0,""""""


In [4]:
# Find rows with empty or null jurisdiction
empty_jurisdiction = df.filter(
    pl.col("Jurisdiction").is_null() | (pl.col("Jurisdiction").str.strip_chars() == "")
)

unique_accessions = empty_jurisdiction.select("Accession").unique()
print(f"Rows with empty jurisdiction: {len(empty_jurisdiction):,}")

print(f"\nUnique accession list({len(unique_accessions)}): ")
print(unique_accessions.to_series().to_list())
empty_jurisdiction.head(10)

Rows with empty jurisdiction: 0

Unique accession list(0): 
[]


Accession,URL,SubsidiaryId,Subsidiary,Jurisdiction,NestingLevel,ParentName,ParentId,Ownership,Footnotes
str,str,str,str,str,i64,str,str,f64,str


## 1. Nested Subsidiaries (NestingLevel >= 1)

In [5]:
# Find nested subsidiaries
nested = df.filter(pl.col("NestingLevel") >= 1)

nested_accessions = nested.select("Accession").unique()
print(f"Nested subsidiaries (level >= 1): {len(nested):,}")

print(f"Unique accession list({len(nested_accessions)}):")
print(nested_accessions.to_series().to_list())
# nested.head(20)


# Filter by specific accession
pl.Config.set_tbl_rows(-1) 
target_accession = '`000007889025000059'
df.filter(pl.col("Accession") == target_accession)


Nested subsidiaries (level >= 1): 6,120
Unique accession list(188):
['`000103130825000002', '`000162828025015673', '`000159696125000024', '`000162828025053871', '`000092480525000012', '`000185224425000007', '`000155943225000035', '`000162828025008121', '`000095017025047891', '`000129281425001352', '`000007169125000047', '`000116486325000009', '`000169913625000014', '`000121390025018782', '`000132611025000033', '`000089542125000304', '`000119312525319187', '`000161755325000011', '`000162828025020368', '`000162828025054103', '`000000497725000047', '`000162828025008730', '`000000620125000010', '`000007889025000059', '`000162828025009448', '`000121390025035542', '`000093905725000159', '`000163720725000016', '`000009212225000018', '`000088461425000053', '`000187901625000003', '`000143774925010228', '`000154565425000005', '`000192956125000044', '`000166028025000034', '`000162828025011831', '`000162828025009503', '`000131415225000031', '`000160548425000013', '`000160402825000012', '`000081158

Accession,URL,SubsidiaryId,Subsidiary,Jurisdiction,NestingLevel,ParentName,ParentId,Ownership,Footnotes
str,str,str,str,str,i64,str,str,f64,str
"""`000007889025000059""","""https://www.sec.gov/Archives/e…","""ead19cab-71d0-5258-9aaf-b3946c…","""The Pittston Company""","""Delaware""",0,null,"""6efe832d-6b6b-5802-a8fb-92caa2…",100.0,""""""
"""`000007889025000059""","""https://www.sec.gov/Archives/e…","""a39d0fa8-f1e5-5d5f-82ac-10df52…","""Glen Allen Development, Inc.""","""Delaware""",0,null,"""6efe832d-6b6b-5802-a8fb-92caa2…",100.0,""""""
"""`000007889025000059""","""https://www.sec.gov/Archives/e…","""cb1c82c2-1818-51f3-bbfa-24431a…","""Liberty National Development C…","""Delaware""",1,"""Glen Allen Development, Inc.""","""a39d0fa8-f1e5-5d5f-82ac-10df52…",32.5,""""""
"""`000007889025000059""","""https://www.sec.gov/Archives/e…","""745520ca-2daf-5858-b812-d146d7…","""New Liberty Residential Urban …","""New Jersey""",1,"""Glen Allen Development, Inc.""","""a39d0fa8-f1e5-5d5f-82ac-10df52…",17.5,""""""
"""`000007889025000059""","""https://www.sec.gov/Archives/e…","""b13312e2-bcfd-521b-83f1-1eabed…","""Pittston Services Group Inc.""","""Virginia""",0,null,"""6efe832d-6b6b-5802-a8fb-92caa2…",100.0,""""""
"""`000007889025000059""","""https://www.sec.gov/Archives/e…","""e8563236-f104-55b5-a23a-68622d…","""Brink’s Holding Company""","""Delaware""",1,"""Pittston Services Group Inc.""","""b13312e2-bcfd-521b-83f1-1eabed…",100.0,""""""
"""`000007889025000059""","""https://www.sec.gov/Archives/e…","""83a69848-1ce7-5a7d-88ed-873563…","""Brink’s Finance Holding Compan…","""Delaware""",2,"""Brink’s Holding Company""","""e8563236-f104-55b5-a23a-68622d…",100.0,""""""
"""`000007889025000059""","""https://www.sec.gov/Archives/e…","""5716b250-b1d1-5923-a92c-015d0c…","""Brink’s Capital Holding Compan…","""Delaware""",3,"""Brink’s Finance Holding Compan…","""83a69848-1ce7-5a7d-88ed-873563…",100.0,""""""
"""`000007889025000059""","""https://www.sec.gov/Archives/e…","""6111f985-879c-5333-a2da-7413ce…","""Brink’s Capital, LLC""","""Delaware""",4,"""Brink’s Capital Holding Compan…","""5716b250-b1d1-5923-a92c-015d0c…",100.0,""""""


In [6]:
# Distribution of nesting levels
df.group_by("NestingLevel").agg(pl.count().alias("count")).sort("NestingLevel")

/var/folders/pg/rrvt9fgx479brk1wjwzw9bqc0000gn/T/ipykernel_42639/269099081.py:2: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  df.group_by("NestingLevel").agg(pl.count().alias("count")).sort("NestingLevel")


NestingLevel,count
i64,u32
0,174450
1,3997
2,1117
3,577
4,216
5,88
6,92
7,18
8,15


In [7]:
# TODO: Find parent rows based on ParentId = SubsidiaryId
# (Commented out - we don't have SubsidiaryId column yet)

# nested_with_parents = nested.join(
#     df.select(["SubsidiaryId", "Subsidiary", "Jurisdiction"]).rename({"Subsidiary": "ParentSubsidiary", "Jurisdiction": "ParentJurisdiction"}),
#     left_on="ParentId",
#     right_on="SubsidiaryId",
#     how="left"
# )
# nested_with_parents

## 2. Rows with Footnotes

In [8]:
# Find rows with non-empty footnotes
with_footnotes = df.filter(
    pl.col("Footnotes").is_not_null() & (pl.col("Footnotes").str.strip_chars() != "")
)

footnotes_accessions = with_footnotes.select("Accession").unique()

print(f"Rows with footnotes: {len(with_footnotes):,}")
print(f"Percentage: {len(with_footnotes) / len(df) * 100:.1f}%")

print(f"\nUnique Accession list({len(footnotes_accessions)}):")
print(footnotes_accessions.to_series().to_list())
with_footnotes.head(20)

Rows with footnotes: 1,436
Percentage: 0.8%

Unique Accession list(202):
['`000081158925000008', '`000095017025042908', '`000121390025026215', '`000121390025043484', '`000117891325001365', '`000143774925004612', '`000007747625000007', '`000102085925000054', '`000103297525000029', '`000132619025000015', '`000100683725000021', '`000121390025043461', '`000164117225001513', '`000095017025030677', '`000121390025021907', '`000007628225000013', '`000110465925108504', '`000132440425000006', '`000162828025006093', '`000094667325000008', '`000084405925000009', '`000110465925123106', '`000141057825000783', '`000157587225000393', '`000009484525000005', '`000010956325000080', '`000153949725002331', '`000121390025037643', '`000000885825000028', '`000095017025045947', '`000118518525000229', '`000162828025011831', '`000149315225011033', '`000155837025001222', '`000105350725000025', '`000143774925005687', '`000152013825000162', '`000008812125000017', '`000164974925000035', '`000008596125000032', '`0000

Accession,URL,SubsidiaryId,Subsidiary,Jurisdiction,NestingLevel,ParentName,ParentId,Ownership,Footnotes
str,str,str,str,str,i64,str,str,f64,str
"""`000149315225014286""","""https://www.sec.gov/Archives/e…","""ae3a6023-d887-54c3-a9a1-d20d99…","""Brigadier Security Systems Ltd…","""Saskatchewan, Canada""",0,null,"""a1d1ae4e-7f40-5345-a808-c76c87…",100.0,"""2000"""
"""`000100683725000021""","""https://www.sec.gov/Archives/e…","""f983a1d0-ab99-5a2b-b1ae-c56bf8…","""DBM Global Inc.""","""Delaware""",0,"""HC2 Broadcasting Inc.""","""03eb6c1c-5330-53db-8db9-e74a9c…",42.35,"""2"""
"""`000100683725000021""","""https://www.sec.gov/Archives/e…","""3ea84c1a-d2dd-5b6d-9350-9ee4c8…","""GrayWolf Industrial, Inc.""","""Delaware""",2,"""CBHorn Holdings, Inc.""","""ef581957-e49b-5501-bf26-4bcfcc…",51.0,"""3, 4"""
"""`000100683725000021""","""https://www.sec.gov/Archives/e…","""d4f565ce-4080-50d5-bc35-f37c57…","""GrayWolf Integrated Constructi…","""Delaware""",3,"""GrayWolf Industrial, Inc.""","""3ea84c1a-d2dd-5b6d-9350-9ee4c8…",69.22,"""5"""
"""`000100683725000021""","""https://www.sec.gov/Archives/e…","""9a321b95-1e2f-59a4-a8a8-a4d473…","""Midwest Environmental, Inc.""","""Kentucky""",3,"""GrayWolf Industrial, Inc.""","""3ea84c1a-d2dd-5b6d-9350-9ee4c8…",100.0,"""6"""
"""`000100683725000021""","""https://www.sec.gov/Archives/e…","""49b48c7b-54c5-5814-ac82-cbaf13…","""Milco National Constructors, I…","""Delaware""",3,"""GrayWolf Industrial, Inc.""","""3ea84c1a-d2dd-5b6d-9350-9ee4c8…",100.0,"""7"""
"""`000100683725000021""","""https://www.sec.gov/Archives/e…","""e046aa9f-b5c3-5f78-9b05-59bfa9…","""DBM Vircon Services , Inc.""","""Arizona""",2,"""DBM Global North America Inc.""","""745dffea-bbc2-5486-af3c-399ef0…",100.0,"""8"""
"""`000100683725000021""","""https://www.sec.gov/Archives/e…","""f8018534-6f22-5888-9318-653587…","""Schuff Steel Company""","""Delaware""",2,"""DBM Global North America Inc.""","""745dffea-bbc2-5486-af3c-399ef0…",100.0,"""9"""
"""`000100683725000021""","""https://www.sec.gov/Archives/e…","""02dfd234-dd92-5703-9b9a-adafbd…","""Derr and Isbell Construction, …","""Texas""",3,"""Schuff Steel Company""","""f8018534-6f22-5888-9318-653587…",100.0,"""10"""


In [9]:
# Unique footnote values per accession
footnotes_by_accession = df.filter(
    pl.col("Footnotes").is_not_null() & (pl.col("Footnotes").str.strip_chars() != "")
).group_by("Accession").agg(
    pl.col("Footnotes").unique().alias("UniqueFootnotes")
)

print(f"Accessions with footnotes: {len(footnotes_by_accession)}")
print(f"\nAccession: Footnotes")
for row in footnotes_by_accession.iter_rows():
    print(f"{row[0]}: {row[1]}")

Accessions with footnotes: 202

Accession: Footnotes
`000007528825000033: ['1', '2', '3', '4', '5', '6', '7', '8']
`000162828025008128: ['01', '02']
`000121390025041556: ['1', '2, 3', '4']
`000093905725000242: ['1', '1, 2']
`000170160525000035: ['1']
`000003069725000003: ['1']
`000094667325000008: ['1', '2']
`000162828025045293: ['2009']
`000000885825000028: ['2008']
`000005125325000013: ['1', '2', '3', '4', '5', '1990', '6', '7', '8']
`000110465925104454: ['1', '2', '3', '4']
`000144889325000009: ['1', '2', '3', '4', '5', '6', '7, 12', '7', '8', '9', '10', '11']
`000197413825000016: ['2010']
`000119312525158722: ['1', '2', '3', '4', '5']
`000110465925108504: ['2']
`000121390025044284: ['1']
`000104013025000072: ['2']
`000003696625000024: ['2', '3']
`000071742325000006: ['1', '1, 2', '2', '3', '4', '5', '6']
`000101376225003608: ['1', '2', '3', '4']
`000106770125000008: ['1', '2']
`000102891825000014: ['2']
`000121390025037703: ['1', '2', '3', '4', '5']
`000162828025019714: ['1']
`0001

In [10]:
# Unique footnote values - extract all individual numbers with their accessions
import re

footnotes_with_accession = df.filter(
    pl.col("Footnotes").is_not_null() & (pl.col("Footnotes").str.strip_chars() != "")
).select(["Accession", "Footnotes"]).unique()

# Build a dict: number -> list of accessions
number_to_accessions = {}
for row in footnotes_with_accession.iter_rows():
    accession, footnote = row
    nums = re.findall(r'\d+', footnote)
    for n in nums:
        num = int(n)
        if num not in number_to_accessions:
            number_to_accessions[num] = set()
        number_to_accessions[num].add(accession)

# Sort by number descending and print
sorted_items = sorted(number_to_accessions.items(), key=lambda x: x[0], reverse=True)

print(f"Total unique footnote numbers: {len(sorted_items)}")
print(f"\nFootnoteNumber: Accessions")
for num, accs in sorted_items:
    print(f"{num}: {list(accs)}")

Total unique footnote numbers: 102

FootnoteNumber: Accessions
25262: ['`000141057825001475']
25093: ['`000141057825001475']
25092: ['`000141057825001475']
2025: ['`000159036425000006']
2024: ['`000162828025005126']
2023: ['`000074026025000052']
2021: ['`000162529725000016']
2018: ['`000007747625000007']
2015: ['`000171218425000031']
2014: ['`000162529725000016', '`000121390025037643', '`000077149725000031']
2013: ['`000117891325000821']
2012: ['`000155118225000006']
2010: ['`000197413825000016', '`000077149725000031']
2009: ['`000162828025045293', '`000121390025043459']
2008: ['`000000885825000028', '`000162828025005311']
2007: ['`000162828025053207']
2005: ['`000006270925000015']
2003: ['`000158536425000014', '`000162828025006093']
2002: ['`000031920125000024']
2001: ['`000121390025020360', '`000095017025027569', '`000093796625000009']
2000: ['`000149315225014286']
1999: ['`000006270925000015']
1998: ['`000155837025001224', '`000117891325000821']
1997: ['`000121390025043461', '`00010